# Emergent Bias in Multi-Agent Climate Budget Allocation
### JupyterLab + Ollama orchestration notebook

Runs the full study pipeline: Ollama serving every agent's model locally
(CPU, Apple Metal, or CUDA GPU -- Ollama supports all three), the 8-agent
negotiation across 3 topologies, and all 3 plot types for whichever
diagnostic study you select below. Everything is saved to Google Drive
(the only persistence backend this project uses) if available -- see the
Drive-mount cell.

**Before running:** upload/clone this entire repository folder into your
JupyterLab workspace so this notebook sits next to `config.py`,
`main.py`, `setup_ollama.py`, etc.


## 1. Check Ollama is installed

In [ ]:
import subprocess, os, sys
print("Working directory:", os.getcwd())
print(subprocess.run(["ls"], capture_output=True, text=True).stdout)

ollama_check = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
if ollama_check.returncode == 0:
    print(ollama_check.stdout)
else:
    print("*** Ollama not found. Install it first: ***")
    print("    Linux:          curl -fsSL https://ollama.com/install.sh | sh")
    print("    macOS/Windows:  download the app from https://ollama.com/download")


## 2. Install Python dependencies
Only needs to run once per environment (skip if already installed).

In [ ]:
!pip install -q -r requirements.txt


## 3. Mount / resolve Google Drive
This uses `drive_utils.py` — in Colab it mounts Drive directly; on a
JupyterLab-on-VM setup it looks for an already-mounted Drive path. If
neither is found, it falls back to a clearly-labeled LOCAL directory and
says so explicitly — it will never silently claim results are on Drive
when they are not. Google Drive is the only place this project saves
code/results/messages, so it's worth getting this working (see README
§8.2 for the rclone walkthrough if you're not on Colab).


In [ ]:
import config, drive_utils

# Optionally hardcode a path instead of auto-detecting, e.g.:
# root = drive_utils.mount_and_resolve_drive_root("/content/drive/MyDrive/climate-mas-bias")
root = drive_utils.mount_and_resolve_drive_root()
config.set_project_root(root)
print("All outputs will be written under:", config.PROJECT_ROOT)


## 4. Choose a diagnostic study

Edit `STUDY_NAME` below. Available studies (defined in `config.STUDY_CONFIGS`
— edit that dict directly to add a 5th study, nothing else needs to change):

| Study | What it isolates |
|---|---|
| `baseline` | All 6 roles on the 4B model (`qwen3.5:4b`), Judge+Improver active |
| `study1_no_judge_improver` | **Ablation** — no Judge, no Improver at all |
| `study2a_all_4b` | All 6 roles on `qwen3.5:4b` |
| `study2b_all_9b` | All 6 roles on `qwen3.5:9b` |
| `study2c_judge_improver_9b` | Residents+functional agents 4B; Judge+Improver upgraded to 9B |
| `study3_heterogeneous` | Different model families per role (`llama3.1:8b`, `mistral:7b-instruct-v0.3`, `qwen3.5:*`) |


In [ ]:
STUDY_NAME = "baseline"  # <-- change this to run a different study

study_cfg = config.STUDY_CONFIGS[STUDY_NAME]
print(f"Study: {STUDY_NAME}\n  {study_cfg['description']}")
print(f"Judge/Improver: {'ENABLED' if study_cfg['use_judge_improver'] else 'DISABLED'}")
for role, key in study_cfg["role_models"].items():
    print(f"  {role:16s} -> {config.ollama_tag_for(key)}")

required = config.required_model_keys(STUDY_NAME)
print(f"\nUnique model tags needed: {[config.ollama_tag_for(k) for k in required]}")


## 5. Pull this study's models with Ollama

Checks the Ollama server is reachable (tries to start it if not), then
runs `ollama pull` for every unique model tag the study needs — skips
tags that are already pulled, so this is safe to re-run.


In [ ]:
import setup_ollama

setup_ollama.start_server_if_needed()
setup_ollama.ensure_models_for_study(STUDY_NAME)


## 6. Run the study
Executes all 3 topologies (Star / Parallel / Ring) for `STUDY_NAME`: every
agent message is printed to the terminal AND written to a full transcript
`.txt` file, Relative Gini is computed after every single message, and
JSON traces + all 3 plot types are saved under `config.PROJECT_ROOT`
(on Google Drive).


In [ ]:
!python main.py --study {STUDY_NAME}


## 7. Display the plots inline

In [ ]:
import glob
from IPython.display import Image, display

for path in sorted(glob.glob(f"{config.FIGURE_DIR}/{STUDY_NAME}__*.png")):
    print(path)
    display(Image(filename=path))


## 8. (Optional) Run more studies, then compare them

Re-run cells 4-7 with a different `STUDY_NAME` (e.g. `study1_no_judge_improver`,
then `study2a_all_4b`, `study2b_all_9b`, `study2c_judge_improver_9b`) — each
run's traces/transcripts/figures are all tagged by study name, so nothing
gets overwritten, and Ollama re-uses any model tags already pulled. Once
you've run the ones you want to compare:


In [ ]:
import evaluate

studies_to_compare = [
    "study1_no_judge_improver",
    "baseline",
    "study2a_all_4b",
    "study2b_all_9b",
    "study2c_judge_improver_9b",
]
averages = evaluate.cross_study_average_gini(studies_to_compare, trace_dir=config.TRACE_DIR)
print(averages)

from logger import plot_cross_study_average_gini
path = plot_cross_study_average_gini(averages, save_path=f"{config.FIGURE_DIR}/cross_study__avg_relative_gini.png")
from IPython.display import Image, display
display(Image(filename=path))


## 9. (Optional) See what's currently loaded / free up memory
`ollama ps` shows which models are currently resident in memory. Ollama
unloads idle models on its own (default after 5 minutes) — you don't
need to manually stop anything between studies unless you want the
memory back immediately.


In [ ]:
!ollama ps
